# Phase 2 — stitched bridge skeleton: build it and prove the plumbing

Plan v2 (`docs/encoder_decoder_bridge_e2e_plan_v2.md`), Phase 2; implementation notes in `docs/stitched-e2e-plan.md`.

```
English source ──► ModernBERT-base (+ LoRA, frozen) ──► bridge: Linear 768→2560 + RMSNorm        [GPU 0]
                                                              │ memory [B, S, 2560] + padding mask
                                                              ▼
persona instruction ──► Gemma 3 4B IT, 4-bit (+ LoRA, frozen)                                       [GPU 1]
   (no source text)       after every 4th layer:  h ← h + tanh(gate) · CrossAttention(norm(h), memory)
                          gates start at 0, so the untouched system *is* Gemma
                                                              ▼
                                                   Hinglish line in the persona's voice
```

- **The source reaches the decoder only through the encoder.** The decoder's prompt is the persona instruction without the sentence, so the decoder cannot bypass the bridge (v2 §2).
- **Cross-attention is attached with forward hooks** on Gemma's decoder layers: Gemma's own modules and weights are untouched, and its loading reports nothing missing because the new modules live outside it.
- **Data:** v3, the three personas kept (Don, Anthony Gonsalves, Jai), with their v4 descriptions: 120 training and 30 validation pairs. Enough for plumbing; the real training data is v4.

**What this notebook must show (v2 Phase 2 gate)**

| # | Test | Pass |
|---|---|---|
| 1 | Shapes and devices; the hooks fire in every chosen layer | memory `[B, S, 2560]` on GPU 1; one call per cross-attention layer per forward |
| 2 | Gate-0 equivalence | with gates at 0, logits equal plain Gemma's |
| 3 | Padding mask | padding a source in a batch does not change its output; removing the mask does |
| 4 | Generation path | cached = uncached tokens; output changes with the gates on, and with another sentence's memory (not a silent fallback to decoder-only) |
| 5 | Gradient audit | gradients reach encoder LoRA, bridge, cross-attention, gates and decoder LoRA; none reach frozen base weights |
| 6 | Memory and speed | measured on both GPUs |
| 7 | Smoke training (bridge + cross-attention + gates only) | loss falls; gates open; validation loss with the right sentence below the no-memory loss, and below the loss with another sentence's memory |

Test 7 is a plumbing check, not a translation result: the decoder and encoder stay frozen and the data is small.

In [ ]:
# Cell 0 — Hugging Face token (Gemma 3 is gated). Clear it before saving or sharing the notebook.

HF_TOKEN = ""

## Cell 1 — Setup

Needs **two GPUs** (Kaggle: *GPU T4 × 2*). Encoder and bridge on `cuda:0`, decoder and cross-attention on `cuda:1`, as in the requirements. Compute in fp32 throughout (Gemma 3 is unstable in fp16; its weights are 4-bit).

In [ ]:
# Cell 1 — Packages, settings, versions, pinned revisions

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2" -U bitsandbytes

import json, math, random, time, urllib.request

import bitsandbytes
import peft
import torch
import torch.nn as nn
import transformers
from huggingface_hub import HfApi, login

ENCODER, DECODER = "answerdotai/ModernBERT-base", "google/gemma-3-4b-it"
DATA = ("https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/"
        "project4-hindi-decoder/data/persona_translation_v3/")
PERSONAS = ["don", "anthony_gonsalves", "jai_sholay"]
XATTN_EVERY, XATTN_HEADS = 4, 8
SEED = 42
DTYPE = torch.float32

assert torch.cuda.device_count() >= 2, "this notebook needs two GPUs (Kaggle: GPU T4 x2)"
ENC_DEVICE, DEC_DEVICE = torch.device("cuda:0"), torch.device("cuda:1")
HF_TOKEN = HF_TOKEN.strip() or None
assert HF_TOKEN, "Gemma 3 is gated: paste a Hugging Face token in Cell 0"
login(HF_TOKEN)

api = HfApi()
REV = {r: api.model_info(r, token=HF_TOKEN).sha for r in (ENCODER, DECODER)}
VERSIONS = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
            "bitsandbytes": bitsandbytes.__version__,
            "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "revisions": REV}
print(json.dumps(VERSIONS, indent=1))

## Cell 2 — Data and the two inputs

- **Encoder input:** the English source sentence, nothing else.
- **Decoder input:** the persona instruction (v4 description), **without the sentence**, through Gemma's chat template; the target is the v3 line plus Gemma's end-of-turn token.

In [ ]:
# Cell 2 — v3 data for the three personas; the encoder and decoder inputs

def fetch(name):
    return json.load(urllib.request.urlopen(DATA + name))


TRAIN, VAL = [[ex for ex in fetch(f"{split}_v3.json") if ex["persona_id"] in PERSONAS]
              for split in ("train", "validation")]

NAME = {"don": "Don", "anthony_gonsalves": "Anthony Gonsalves", "jai_sholay": "Jai (Sholay)"}
DESC = {
    "don": "Cool, calculating and faintly amused; short, controlled sentences; treats every situation as a move "
           "in a game; a quiet hint of control, never loud.",
    "anthony_gonsalves": "Cheerful, theatrical Bombay street-smart; playful exaggeration and comic comparisons; "
                         "freely mixes English and Hindi.",
    "jai_sholay": "Calm, laconic and practical; dry understatement; loyal; says only what is needed.",
}
INSTRUCTION = ("Translate the English sentence you are given into natural Roman-script Hinglish in the voice of "
               "{name}. Persona: {desc} Preserve the original meaning; do not add new events or facts.")


def decoder_prompt(persona):
    return INSTRUCTION.format(name=NAME[persona], desc=DESC[persona])


assert not any(ex["source"] in decoder_prompt(ex["persona_id"]) for ex in TRAIN + VAL)
SOURCE_IDS = sorted({ex["source_id"] for ex in VAL})
print(f"train {len(TRAIN)} pairs | validation {len(VAL)} pairs | personas {PERSONAS}")
print("encoder input:", TRAIN[0]["source"])
print("decoder input:", decoder_prompt(TRAIN[0]["persona_id"]))
print("target:       ", TRAIN[0]["target"])

## Cell 3 — Load the encoder (GPU 0) and the decoder (GPU 1)

The decoder's layer stack is found by name (`…language_model.layers`), and its width and depth are read from its configuration, so the bridge sizes are not hard-coded.

In [ ]:
# Cell 3 — ModernBERT-base on GPU 0; Gemma 3 4B IT (4-bit NF4, fp32 compute) on GPU 1

from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig, Gemma3ForConditionalGeneration

enc_tok = AutoTokenizer.from_pretrained(ENCODER, revision=REV[ENCODER])
enc_cfg = AutoConfig.from_pretrained(ENCODER, revision=REV[ENCODER])
if hasattr(enc_cfg, "reference_compile"):
    enc_cfg.reference_compile = False        # no torch.compile of the embeddings on a T4
encoder = AutoModel.from_pretrained(ENCODER, revision=REV[ENCODER], config=enc_cfg, dtype=DTYPE,
                                    attn_implementation="sdpa").to(ENC_DEVICE).eval()

dec_tok = AutoTokenizer.from_pretrained(DECODER, revision=REV[DECODER], token=HF_TOKEN)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
decoder = Gemma3ForConditionalGeneration.from_pretrained(DECODER, revision=REV[DECODER], dtype=DTYPE, token=HF_TOKEN,
                                                         quantization_config=quant, device_map={"": DEC_DEVICE.index},
                                                         attn_implementation="eager")
TEXT = decoder.config.text_config
D_ENC, D_DEC, N_LAYERS = encoder.config.hidden_size, TEXT.hidden_size, TEXT.num_hidden_layers
LAYERS = next(m for n, m in decoder.named_modules() if n.endswith("language_model.layers"))
assert len(LAYERS) == N_LAYERS
eos = decoder.generation_config.eos_token_id
STOPS = sorted(set(eos if isinstance(eos, list) else [eos]))
PAD = dec_tok.pad_token_id

print(f"encoder width {D_ENC} | decoder width {D_DEC}, {N_LAYERS} layers | stop tokens {STOPS}")
print(f"GPU 0: {torch.cuda.memory_allocated(ENC_DEVICE) / 2**30:.1f} GB | "
      f"GPU 1: {torch.cuda.memory_allocated(DEC_DEVICE) / 2**30:.1f} GB")

## Cell 4 — Bridge, gated cross-attention, and the hooks that attach them

- **Bridge** (GPU 0): `Linear(768 → 2560)` then RMSNorm, so the memory has a stable scale whatever the encoder produces.
- **Gated cross-attention** (GPU 1), one block per chosen layer: pre-norm `nn.MultiheadAttention` (queries from the decoder, keys and values from the memory, encoder padding masked), added to the layer's output through `tanh(gate)`, **gate initialised to 0**.
- **Hooks:** a forward hook on each chosen Gemma layer replaces the layer's output `h` with `h + tanh(gate) · CrossAttention(h, memory)`. The memory and its padding mask are set once per batch in a small context object; the hooks read it at every forward, including every generation step. The context also counts hook calls, which the tests use.

In [ ]:
# Cell 4 — Bridge, gated cross-attention blocks, hooks, and the stitched forward

class Bridge(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.proj = nn.Linear(d_in, d_out)
        self.norm = nn.RMSNorm(d_out)

    def forward(self, h):
        return self.norm(self.proj(h))


class GatedCrossAttention(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.norm = nn.RMSNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, h, memory, pad):
        out, _ = self.attn(self.norm(h), memory, memory, key_padding_mask=pad, need_weights=False)
        return h + torch.tanh(self.gate) * out


class Context:
    enabled, memory, pad, calls = True, None, None, 0


CTX = Context()
XATTN_LAYERS = list(range(XATTN_EVERY - 1, N_LAYERS, XATTN_EVERY))
torch.manual_seed(SEED)
bridge = Bridge(D_ENC, D_DEC).to(ENC_DEVICE)
xattn = nn.ModuleDict({str(i): GatedCrossAttention(D_DEC, XATTN_HEADS) for i in XATTN_LAYERS}).to(DEC_DEVICE)


def make_hook(i):
    block = xattn[str(i)]

    def hook(module, args, output):
        if not CTX.enabled:
            return output
        CTX.calls += 1
        h = output[0] if isinstance(output, tuple) else output
        h = block(h, CTX.memory, CTX.pad)
        return (h, *output[1:]) if isinstance(output, tuple) else h

    return hook


HOOKS = [LAYERS[i].register_forward_hook(make_hook(i)) for i in XATTN_LAYERS]


def encode(sources, train_encoder=False):
    """English sources -> bridge memory on GPU 1 and its padding mask (True = padding)."""
    t = enc_tok(sources, padding=True, truncation=True, max_length=128, return_tensors="pt").to(ENC_DEVICE)
    with torch.set_grad_enabled(train_encoder and torch.is_grad_enabled()):
        h = encoder(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).last_hidden_state
    return bridge(h).to(DEC_DEVICE), (t["attention_mask"] == 0).to(DEC_DEVICE)


def set_memory(memory, pad, enabled=True):
    CTX.memory, CTX.pad, CTX.enabled = memory, pad, enabled


def prompt_text(persona):
    msgs = [{"role": "user", "content": decoder_prompt(persona)}]
    return dec_tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)


def encode_pair(ex):
    msgs = [{"role": "user", "content": decoder_prompt(ex["persona_id"])},
            {"role": "assistant", "content": ex["target"]}]
    full, head = dec_tok.apply_chat_template(msgs, tokenize=False), prompt_text(ex["persona_id"])
    assert full.startswith(head)
    prompt = dec_tok(head, add_special_tokens=False)["input_ids"]
    reply = dec_tok(full[len(head):], add_special_tokens=False)["input_ids"]
    assert any(t in STOPS for t in reply)
    return torch.tensor(prompt + reply), torch.tensor([-100] * len(prompt) + reply)


def decoder_batch(examples):
    pairs = [encode_pair(ex) for ex in examples]
    T = max(len(x) for x, _ in pairs)
    ids = torch.full((len(pairs), T), PAD)
    labels = torch.full((len(pairs), T), -100)
    mask = torch.zeros((len(pairs), T), dtype=torch.long)
    for i, (x, y) in enumerate(pairs):
        ids[i, :len(x)], labels[i, :len(y)], mask[i, :len(x)] = x, y, 1
    return {"input_ids": ids.to(DEC_DEVICE), "attention_mask": mask.to(DEC_DEVICE), "labels": labels.to(DEC_DEVICE),
            "token_type_ids": torch.zeros_like(ids).to(DEC_DEVICE)}


print(f"cross-attention after decoder layers {XATTN_LAYERS} ({len(XATTN_LAYERS)} blocks)")
print(f"bridge {sum(p.numel() for p in bridge.parameters()) / 1e6:.1f} M parameters | "
      f"cross-attention {sum(p.numel() for p in xattn.parameters()) / 1e6:.1f} M")

## Cell 5 — LoRA at both ends (created, frozen) and the parameter groups

Plan v2: both adapters are part of the architecture, but in Phase 2 they are only instantiated; full adapter training starts in Phase 3. Encoder LoRA on ModernBERT's `Wqkv`, `Wo`, `Wi`; decoder LoRA on Gemma's attention and MLP projections (language model only). `prepare_model_for_kbit_training` sets up the 4-bit decoder for gradients (with gradient checkpointing).

Five parameter groups are tracked separately throughout: encoder LoRA, bridge, cross-attention, gates, decoder LoRA.

In [ ]:
# Cell 5 — Encoder and decoder LoRA; parameter groups; trainable = bridge + cross-attention + gates

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

decoder = prepare_model_for_kbit_training(decoder, use_gradient_checkpointing=True,
                                          gradient_checkpointing_kwargs={"use_reentrant": False})
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
dec_model = get_peft_model(decoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                               target_modules=rf".*language_model.*\.({'|'.join(PROJ)})"))
encoder = get_peft_model(encoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                             target_modules=["Wqkv", "Wo", "Wi"]))
assert all(LAYERS[i]._forward_hooks for i in XATTN_LAYERS), "the hooks did not survive the LoRA wrapping"

GROUPS = {
    "encoder LoRA": [p for n, p in encoder.named_parameters() if "lora_" in n],
    "bridge": list(bridge.parameters()),
    "cross-attention": [p for n, p in xattn.named_parameters() if not n.endswith("gate")],
    "gates": [b.gate for b in xattn.values()],
    "decoder LoRA": [p for n, p in dec_model.named_parameters() if "lora_" in n],
}


def set_trainable(*names):
    for name, params in GROUPS.items():
        for p in params:
            p.requires_grad_(name in names)


def set_gates(value):
    for b in xattn.values():
        b.gate.data.fill_(value)


SMOKE_GROUPS = ("bridge", "cross-attention", "gates")
set_trainable(*SMOKE_GROUPS)
for name, params in GROUPS.items():
    print(f"  {name:16s} {sum(p.numel() for p in params) / 1e6:8.2f} M parameters  "
          f"{'trainable' if name in SMOKE_GROUPS else 'frozen'}")

## Cell 6 — Tests 1–3: shapes and hooks, gate-0 equivalence, padding mask

In [ ]:
# Cell 6 — Test 1 (shapes, devices, hook calls), Test 2 (gate-0 equivalence), Test 3 (padding mask)

TESTS = {}
dec_model.eval()
encoder.eval()


def logits_for(examples, sources, enabled=True, pad_override=False):
    dec = decoder_batch(examples)
    dec.pop("labels")
    memory, pad = encode(sources)
    set_memory(memory, None if pad_override else pad, enabled)
    CTX.calls = 0
    return dec_model(**dec).logits, memory, pad


with torch.no_grad():
    # Test 1
    logits, memory, pad = logits_for(VAL[:4], [ex["source"] for ex in VAL[:4]])
    TESTS["1 shapes, devices, hooks"] = (memory.shape[-1] == D_DEC and memory.device == DEC_DEVICE
                                        and CTX.calls == len(XATTN_LAYERS))
    print(f"Test 1: memory {tuple(memory.shape)} on {memory.device}, padding mask {tuple(pad.shape)}, "
          f"hook calls per forward {CTX.calls} (expected {len(XATTN_LAYERS)})")

    # Test 2: with gates at 0 the stitched model is Gemma
    set_gates(0.0)
    plain, _, _ = logits_for(VAL[:4], [ex["source"] for ex in VAL[:4]], enabled=False)
    stitched, _, _ = logits_for(VAL[:4], [ex["source"] for ex in VAL[:4]])
    diff = (plain - stitched).abs().max().item()
    TESTS["2 gate-0 equivalence"] = diff <= 1e-5
    print(f"Test 2: max |logit difference| plain vs stitched with gates at 0 = {diff:.2e} (pass <= 1e-5)")

    # Test 3: padding the encoder input must not change the output (gates open so the memory matters)
    set_gates(1.0)
    n_tokens = lambda ex: len(enc_tok(ex["source"])["input_ids"])
    a, longer = min(VAL, key=n_tokens), max(VAL, key=n_tokens)
    assert n_tokens(longer) > n_tokens(a)
    alone, _, _ = logits_for([a], [a["source"]])
    batched, _, pad = logits_for([a, a], [a["source"], longer["source"]])
    unmasked, _, _ = logits_for([a, a], [a["source"], longer["source"]], pad_override=True)
    d_masked = (alone[0] - batched[0]).abs().max().item()
    d_unmasked = (alone[0] - unmasked[0]).abs().max().item()
    TESTS["3 padding mask"] = d_masked <= 1e-3 and d_unmasked > 100 * max(d_masked, 1e-6)
    print(f"Test 3: source padded with {int(pad[0].sum())} positions | difference with mask {d_masked:.2e} "
          f"(pass <= 1e-3) | without mask {d_unmasked:.2e} (must be clearly larger)")
    set_gates(0.0)

## Cell 7 — Test 4: the generation path

With the gates open (tanh(1) ≈ 0.76) and untrained cross-attention, the output is not meant to be good; it is meant to be **deterministic and dependent on the memory**:

- cached and uncached generation give the same tokens;
- the hooks fire at every generation step;
- switching the cross-attention off, or giving another sentence's memory, changes the output.

In [ ]:
# Cell 7 — Test 4: cached vs uncached generation; the memory is used at every step

@torch.no_grad()
def generate(ex, memory_source=None, enabled=True, use_cache=True, max_new_tokens=24):
    ids = dec_tok(prompt_text(ex["persona_id"]), return_tensors="pt", add_special_tokens=False).to(DEC_DEVICE)
    memory, pad = encode([memory_source or ex["source"]])
    set_memory(memory, pad, enabled)
    CTX.calls = 0
    out = dec_model.generate(**ids, do_sample=False, num_beams=1, max_new_tokens=max_new_tokens,
                             eos_token_id=STOPS, pad_token_id=PAD, use_cache=use_cache,
                             temperature=None, top_p=None, top_k=None)
    new = out[0, ids["input_ids"].shape[1]:].tolist()
    return new, CTX.calls


set_gates(1.0)
ex, other = VAL[0], next(e for e in VAL if e["source_id"] != VAL[0]["source_id"])
cached, calls = generate(ex)
uncached, _ = generate(ex, use_cache=False)
off, _ = generate(ex, enabled=False)
swapped, _ = generate(ex, memory_source=other["source"])
set_gates(0.0)

every_step = calls % len(XATTN_LAYERS) == 0 and calls // len(XATTN_LAYERS) >= len(cached)
TESTS["4 generation path"] = cached == uncached and every_step and cached != off and cached != swapped
print(f"cached == uncached: {cached == uncached} | hook calls {calls} = {len(XATTN_LAYERS)} layers x "
      f"{calls // len(XATTN_LAYERS)} forward passes for {len(cached)} new tokens: {every_step}")
print(f"differs with cross-attention off: {cached != off} | differs with another sentence's memory: {cached != swapped}")
for label, toks in [("gates 0.5", cached), ("cross-attention off", off), ("other memory", swapped)]:
    print(f"  {label:20s} {dec_tok.decode(toks, skip_special_tokens=True)!r}")

## Cell 8 — Test 5: gradient audit (all five groups)

All five groups are made trainable for one step, the gates set to 0.1 so gradients can pass through them, and one training batch is run forward and backward with the encoder in the graph. Each group must receive a non-zero gradient; no frozen base weight may receive one. (LoRA's `B` matrices start at zero, so in each LoRA group only `B` receives a gradient on the first step; the group norm is still non-zero.)

In [ ]:
# Cell 8 — Test 5: gradient norms per group; frozen weights must get none

def stitched_loss(examples, memory_sources=None, enabled=True, train_encoder=False):
    dec = decoder_batch(examples)
    memory, pad = encode(memory_sources or [ex["source"] for ex in examples], train_encoder)
    set_memory(memory, pad, enabled)
    return dec_model(**dec).loss


set_trainable(*GROUPS)
set_gates(0.1)
dec_model.train()
for params in GROUPS.values():
    for p in params:
        p.grad = None
stitched_loss(TRAIN[:2], train_encoder=True).backward()

NORMS = {name: math.sqrt(sum(float(p.grad.norm()) ** 2 for p in params if p.grad is not None))
         for name, params in GROUPS.items()}
tracked = {id(p) for params in GROUPS.values() for p in params}
stray = [n for model in (encoder, dec_model) for n, p in model.named_parameters()
         if id(p) not in tracked and p.grad is not None]
TESTS["5 gradient audit"] = all(v > 0 for v in NORMS.values()) and not stray
for name, v in NORMS.items():
    print(f"  {name:16s} gradient norm {v:.3e}")
print(f"frozen base weights with a gradient: {len(stray)}")

for params in GROUPS.values():
    for p in params:
        p.grad = None
set_gates(0.0)
set_trainable(*SMOKE_GROUPS)
dec_model.eval()

## Cell 9 — Test 7: smoke training (bridge + cross-attention + gates)

Only the bridge, the cross-attention blocks and their gates train; encoder, decoder and both LoRAs stay frozen. The gates have their own, larger learning rate so they can open within a short run (they start at exactly 0, and while they are 0 only they receive gradients).

After every epoch, three validation losses on the same 30 pairs:

- **right memory:** each pair with its own sentence;
- **other memory:** each pair with the memory of a sentence from a *different* source (fixed pairing);
- **no memory:** cross-attention off — Gemma with the persona instruction only (it never sees the sentence).

If the bridge carries the sentence, *right* falls below both *no memory* and *other memory*.

In [ ]:
# Cell 9 — Smoke training with the three validation losses each epoch

EPOCHS, MICRO, ACCUM, LR, LR_GATES = 3, 2, 2, 1e-4, 1e-2
opt = torch.optim.AdamW([{"params": GROUPS["bridge"] + GROUPS["cross-attention"], "lr": LR},
                         {"params": GROUPS["gates"], "lr": LR_GATES}], weight_decay=0.0)
OTHER = {sid: SOURCE_IDS[(k + 1) % len(SOURCE_IDS)] for k, sid in enumerate(SOURCE_IDS)}
SOURCE_TEXT = {ex["source_id"]: ex["source"] for ex in VAL}


@torch.no_grad()
def validation_losses(batch=3):
    dec_model.eval()
    out = {"right": 0.0, "other": 0.0, "none": 0.0}
    for i in range(0, len(VAL), batch):
        chunk = VAL[i:i + batch]
        others = [SOURCE_TEXT[OTHER[ex["source_id"]]] for ex in chunk]
        out["right"] += stitched_loss(chunk).item() * len(chunk)
        out["other"] += stitched_loss(chunk, memory_sources=others).item() * len(chunk)
        out["none"] += stitched_loss(chunk, enabled=False).item() * len(chunk)
    return {k: round(v / len(VAL), 4) for k, v in out.items()}


def gate_values():
    return [round(math.tanh(b.gate.item()), 4) for b in xattn.values()]


LOG = [{"epoch": 0, "train_loss": None, **validation_losses(), "gates": gate_values()}]
print(LOG[-1])
rng = random.Random(SEED)
torch.cuda.reset_peak_memory_stats(ENC_DEVICE)
torch.cuda.reset_peak_memory_stats(DEC_DEVICE)
t0 = time.perf_counter()
steps = 0
for epoch in range(1, EPOCHS + 1):
    dec_model.train()
    order = rng.sample(TRAIN, len(TRAIN))
    batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
    losses = []
    for n, b in enumerate(batches, 1):
        loss = stitched_loss(b)
        assert torch.isfinite(loss), "non-finite loss"
        (loss / ACCUM).backward()
        losses.append(loss.item())
        if n % ACCUM == 0 or n == len(batches):
            opt.step()
            opt.zero_grad()
            steps += 1
    LOG.append({"epoch": epoch, "train_loss": round(sum(losses) / len(losses), 4), **validation_losses(),
                "gates": gate_values()})
    print(LOG[-1], f"| {time.perf_counter() - t0:.0f}s")

SECONDS_PER_STEP = (time.perf_counter() - t0) / steps
PEAK = {"GPU 0": round(torch.cuda.max_memory_allocated(ENC_DEVICE) / 2**30, 2),
        "GPU 1": round(torch.cuda.max_memory_allocated(DEC_DEVICE) / 2**30, 2)}

## Cell 10 — Test 6 and the smoke-training checks; sample outputs

Memory and speed (Test 6), the smoke-training checks (Test 7), and a few generations after the smoke run: each validation sentence with its own memory, and with another sentence's memory. They are not expected to be good translations yet; they show whether the output starts to follow the sentence.

In [ ]:
# Cell 10 — Test 6 (memory, speed), Test 7 checks, sample generations

first, last = LOG[0], LOG[-1]
TESTS["6 memory and speed"] = True
TESTS["7 smoke training"] = (last["train_loss"] is not None and LOG[1]["train_loss"] > last["train_loss"]
                             and max(abs(g) for g in last["gates"]) > 0.05
                             and last["right"] < last["none"] and last["right"] < last["other"])
print(f"Test 6: peak memory {PEAK} | {SECONDS_PER_STEP:.2f} s per optimizer step (batch {MICRO * ACCUM})")
print(f"Test 7: training loss {LOG[1]['train_loss']} -> {last['train_loss']} | gates (tanh) {last['gates']}")
print(f"        validation: right memory {last['right']} | other memory {last['other']} | "
      f"no memory {last['none']} (epoch 0: {first['right']})")

SAMPLES = []
for sid in SOURCE_IDS[:2]:
    for ex in [e for e in VAL if e["source_id"] == sid]:
        right, _ = generate(ex, max_new_tokens=48)
        other, _ = generate(ex, memory_source=SOURCE_TEXT[OTHER[sid]], max_new_tokens=48)
        SAMPLES.append({"source_id": sid, "persona": ex["persona_id"], "source": ex["source"],
                        "right_memory": dec_tok.decode(right, skip_special_tokens=True).strip(),
                        "other_memory": dec_tok.decode(other, skip_special_tokens=True).strip(),
                        "reference": ex["target"]})
for s in SAMPLES:
    print(f"\n{s['source_id']} {s['persona']} | English: {s['source']}")
    print(f"  right memory | {s['right_memory']}")
    print(f"  other memory | {s['other_memory']}   (memory of: {SOURCE_TEXT[OTHER[s['source_id']]]})")
    print(f"  reference    | {s['reference']}")

## Cell 11 — Phase 2 gate: summary and settings

In [ ]:
# Cell 11 — All tests, pass / fail, and the settings as JSON

print("PHASE 2 GATE")
for name, ok in TESTS.items():
    print(f"  {'PASS' if ok else 'FAIL'}  Test {name}")
print("\nall pass" if all(TESTS.values()) else "\nnot all tests pass: fix before Phase 3")

SETTINGS = {
    "versions": VERSIONS,
    "encoder": {"model": ENCODER, "width": D_ENC, "lora": {"r": 16, "alpha": 32, "targets": ["Wqkv", "Wo", "Wi"]}},
    "decoder": {"model": DECODER, "width": D_DEC, "layers": N_LAYERS, "precision": "4-bit NF4, fp32 compute",
                "lora": {"r": 16, "alpha": 32, "targets": PROJ}},
    "bridge": "Linear + RMSNorm", "cross_attention": {"after_layers": XATTN_LAYERS, "heads": XATTN_HEADS,
                                                     "gate": "tanh, initialised to 0"},
    "devices": {"encoder + bridge": str(ENC_DEVICE), "decoder + cross-attention": str(DEC_DEVICE)},
    "smoke": {"trainable": list(SMOKE_GROUPS), "epochs": EPOCHS, "micro_batch": MICRO, "accumulation": ACCUM,
              "lr": LR, "lr_gates": LR_GATES, "log": LOG, "peak_memory_gb": PEAK,
              "seconds_per_step": round(SECONDS_PER_STEP, 2)},
    "gradient_norms": NORMS,
    "tests": TESTS,
    "samples": SAMPLES,
}
print("\n----- SETTINGS -----")
print(json.dumps(SETTINGS, ensure_ascii=False, indent=1, default=str))

---
## Next

Save the notebook (clear the token in Cell 0 first).

- **All tests pass:** the plumbing is proven; Phase 3 trains encoder LoRA, bridge, cross-attention and decoder LoRA together on the v4 data.
- **Test 7 fails but 1–6 pass:** the plumbing works but the bridge does not learn in a short smoke run; look at the gate values and the three validation losses before changing anything (longer run, gate learning rate, more cross-attention layers).
- **Any of 1–6 fails:** fix it first; Phase 3 builds on all of them.